<a href="https://colab.research.google.com/github/aartinangare1745/GuardPrompts/blob/main/rules.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re
injection_patterns=[
    r"ignore (all | any)?(previous|prior|above) instructions",
    r"disregard (all |any )?(previous|prior|above) instructions",
    r"you are now (a |an )?",
    r"forget (all |your )?(previous |prior )?instructions",
    r"reveal (your |the )?(system prompt|instructions)",
    r"i am the admin",
    r"jailbreak",
    r"act as (dan|an unrestricted|a jailbroken)",
    r"new instructions:",
]

compiled=[re.compile(p,re.IGNORECASE) for p in injection_patterns]

def rule_chk(prompt:str)->dict:
  for pat in compiled:
    if pat.search(prompt):
      return {"blocked":True,"stage":"rules","matched":prompt}
    return {"blocked":False,"stage":"rules"}

In [7]:
import pandas as pd
test_df=pd.read_csv("/content/drive/MyDrive/guardprompt/data/test.csv")

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
test_df["rule_blocked"] = test_df["prompt"].map(lambda p: rule_chk(p)["blocked"])
print(pd.crosstab(test_df["label"], test_df["rule_blocked"]))

rule_blocked  False  True 
label                     
0              4329      0
1              2742     39


In [7]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score, recall_score, precision_score

X = np.load("/content/drive/MyDrive/guardprompt/data/X_train_emb.npy")
y = np.load("/content/drive/MyDrive/guardprompt/data/y_train.npy")
train_df = pd.read_csv("/content/drive/MyDrive/guardprompt/data/train.csv")

# validation split from TRAIN only
Xtr, Xva, ytr, yva = train_test_split(
    X, y, test_size=0.2, random_state=42,
    stratify=train_df["label"].astype(str) + "_" + train_df["source"]
)

candidates = {
    "logreg_C0.1": LogisticRegression(C=0.1, max_iter=2000, class_weight="balanced"),
    "logreg_C1":   LogisticRegression(C=1,   max_iter=2000, class_weight="balanced"),
    "logreg_C10":  LogisticRegression(C=10,  max_iter=2000, class_weight="balanced"),
    "mlp":         MLPClassifier(hidden_layer_sizes=(256, 64), early_stopping=True, random_state=42),
    "hgb":         HistGradientBoostingClassifier(class_weight="balanced", random_state=42),
}

for name, m in candidates.items():
    m.fit(Xtr, ytr)
    p = m.predict(Xva)
    print(f"{name:12s} P={precision_score(yva,p):.3f} R={recall_score(yva,p):.3f} F1={f1_score(yva,p):.3f}")

logreg_C0.1  P=0.795 R=0.867 F1=0.829
logreg_C1    P=0.819 R=0.891 F1=0.853
logreg_C10   P=0.823 R=0.892 F1=0.856
mlp          P=0.931 R=0.912 F1=0.921
hgb          P=0.858 R=0.880 F1=0.869


In [13]:
from sklearn.preprocessing import normalize
Xtr_n, Xte_n = normalize(np.load("/content/drive/MyDrive/guardprompt/data/X_train_emb.npy")), normalize(np.load("/content/drive/MyDrive/guardprompt/data/X_test_emb.npy"))
max_sim = np.concatenate([(Xte_n[i:i+500] @ Xtr_n.T).max(axis=1) for i in range(0, len(Xte_n), 500)])
for t in [0.90, 0.95, 0.98]:
    print(f"test rows with train neighbor >= {t}: {(max_sim >= t).mean():.1%}")

test rows with train neighbor >= 0.9: 18.1%
test rows with train neighbor >= 0.95: 13.5%
test rows with train neighbor >= 0.98: 9.6%


In [8]:
mlp = MLPClassifier(hidden_layer_sizes=(256, 64), early_stopping=True, random_state=42)
mlp.fit(X, y)   # full train set
pred = mlp.predict(np.load("/content/drive/MyDrive/guardprompt/data/X_test_emb.npy"))
y_test = np.load("/content/drive/MyDrive/guardprompt/data/y_test.npy")
test_df = pd.read_csv("/content/drive/MyDrive/guardprompt/data/test.csv")
print("overall F1:", f1_score(y_test, pred))
for s in test_df.source.unique():
    m = (test_df.source == s).values
    print(s, round(f1_score(y_test[m], pred[m]), 3), m.sum())

overall F1: 0.9260798843303814
NeCent 0.887 5890
MindGard 0.988 1114
Deepset 0.781 106


In [4]:
# mlp may memorizing the leakage

import numpy as np, pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.preprocessing import normalize

X_tr = np.load("/content/drive/MyDrive/guardprompt/data/X_train_emb.npy"); y_tr = np.load("/content/drive/MyDrive/guardprompt/data/y_train.npy")
X_te = np.load("/content/drive/MyDrive/guardprompt/data/X_test_emb.npy");  y_te = np.load("/content/drive/MyDrive/guardprompt/data/y_test.npy")
test_df = pd.read_csv("/content/drive/MyDrive/guardprompt/data/test.csv")

# leakage mask, computed on normalized COPIES (X_tr / X_te stay untouched)
A, B = normalize(X_tr), normalize(X_te)
max_sim = np.concatenate([(B[i:i+500] @ A.T).max(axis=1) for i in range(0, len(B), 500)])
clean = max_sim < 0.95

models = {
    "logreg": LogisticRegression(C=10, max_iter=2000, class_weight="balanced"),
    "mlp": MLPClassifier(hidden_layer_sizes=(256, 64), early_stopping=True, random_state=42),
}
for name, m in models.items():
    m.fit(X_tr, y_tr)
    p = m.predict(X_te)
    print(f"\n{name}: all={f1_score(y_te,p):.3f}  clean={f1_score(y_te[clean],p[clean]):.3f}  leaky={f1_score(y_te[~clean],p[~clean]):.3f}")
    for s in test_df.source.unique():
        k = (test_df.source == s).values & clean
        print(f"  {s} clean: {f1_score(y_te[k],p[k]):.3f} (n={k.sum()})")


logreg: all=0.854  clean=0.807  leaky=0.980
  NeCent clean: 0.774 (n=5597)
  MindGard clean: 0.943 (n=472)
  Deepset clean: 0.480 (n=80)

mlp: all=0.926  clean=0.899  leaky=0.992
  NeCent clean: 0.879 (n=5597)
  MindGard clean: 0.973 (n=472)
  Deepset clean: 0.787 (n=80)


In [9]:
import os
import joblib

os.makedirs("models", exist_ok=True)
joblib.dump(mlp, "models/mlp.joblib")

['models/mlp.joblib']

In [10]:
import joblib
import numpy as np

# Load the saved model directly from the file
loaded_mlp = joblib.load("models/mlp.joblib")

# Load test embeddings for a quick test
X_te = np.load("/content/drive/MyDrive/guardprompt/data/X_test_emb.npy")


predictions = loaded_mlp.predict(X_te)
print(f"Loaded model successfully! Generated {len(predictions)} predictions.")

Loaded model successfully! Generated 7110 predictions.


Caliberation check

In [12]:
import numpy as np, pandas as pd, joblib
from sklearn.neural_network import MLPClassifier


mlp = MLPClassifier(hidden_layer_sizes=(256, 64), early_stopping=True, random_state=42).fit(X_tr, y_tr)
joblib.dump(mlp, "models/mlp.joblib")

proba = mlp.predict_proba(X_te)[:, 1]
wrong = (proba >= 0.5).astype(int) != y_te

for lo, hi in [(0.4, 0.6), (0.3, 0.7), (0.2, 0.8)]:
    band = (proba >= lo) & (proba <= hi)
    print(f"band {lo}-{hi}: {band.mean():.1%} of prompts, "
          f"{(band & wrong).sum()/wrong.sum():.1%} of all errors, "
          f"accuracy inside band {1 - wrong[band].mean():.1%}")

band 0.4-0.6: 2.4% of prompts, 17.6% of all errors, accuracy inside band 57.1%
band 0.3-0.7: 4.5% of prompts, 33.3% of all errors, accuracy inside band 57.6%
band 0.2-0.8: 7.2% of prompts, 49.4% of all errors, accuracy inside band 60.6%
